# Component 04 — PP1 training notebook
Train one Random Forest on **manually reviewed five-second vehicle-pair windows**. YOLO/ByteTrack provides trajectories; accident/normal folders are not four-state labels. No model or fabricated results are prefilled.

Run top to bottom. GPU is optional for extraction; CPU is sufficient for Random Forest. Use Streamlit locally for annotation, then upload its **labelled_features.csv** export. All training/inference functions are shared with the Windows app.


In [ ]:
# 1. Upload the supplied Component04_PP1.zip (trusted project code).
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next((n for n in uploaded if n.endswith('.zip')), None)
assert archive, 'Upload Component04_PP1.zip'
work=Path('/content/component04'); work.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert (work/name).resolve().is_relative_to(work.resolve()), 'Unsafe archive path'
    z.extractall(work)
project=next(work.rglob('risk_core.py')).parent
os.chdir(project)
print(project)


In [ ]:
# 2. Install and record exact environment versions.
%pip -q install -r requirements.txt
import subprocess, sys
with open('installed_versions.txt','w') as output:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=output,check=True)


In [ ]:
# 3. Mount Drive; edit the output directory if desired.
from google.colab import drive
drive.mount('/content/drive')
OUTPUT=Path('/content/drive/MyDrive/Component04_PP1')
OUTPUT.mkdir(parents=True,exist_ok=True)


## Optional GPU video extraction
Skip if footage has already been extracted locally. Set the parent folder containing the accident/normal videos and enable the cell. Folder names are deliberately ignored as risk labels. Copy the resulting `runs/<hash>` folders into your Windows package's `data/runs/`, then upload the **same original videos** in Streamlit to review/annotate the extracted windows. Hashes match only for identical video bytes.


In [ ]:
# 4. OPTIONAL: Runtime → Change runtime type → GPU, if available.
RUN_EXTRACTION=False
VIDEO_FOLDER=Path('/content/drive/MyDrive/CHANGE_TO_YOUR_VIDEO_FOLDER')
if RUN_EXTRACTION:
    from video_pipeline import extract_video
    videos=sorted(p for p in VIDEO_FOLDER.rglob('*') if p.suffix.lower() in ['.mp4','.avi','.mov','.mkv'])
    assert videos, 'Correct VIDEO_FOLDER first'
    for video in videos:
        tracks,features,metadata,out=extract_video(video,OUTPUT/'runs',max_seconds=120)
        print(video.name,len(features),'eligible pair windows',out)


## Annotation and cleaning
Use Streamlit's annotation guide: Normal, Increasing Risk, Developing Conflict, Critical Conflict / Collision. Review vehicle IDs and multiple visible cues. Label the pair's state at window end; do not use future crash frames to relabel an earlier state. Skip unclear cases and track-ID switches.

Export **labelled_features.csv for Colab**. Clips from one original recording must share `source_group`; overlapping windows must never cross the split. Do not derive ground-truth states using the same geometric threshold you want a model to predict. Ideally have a second reviewer check a subset.


In [ ]:
# 5. Upload the joined feature + annotation table from Streamlit.
import pandas as pd
from risk_core import FEATURES,STATES,train_model
uploads=files.upload()
csv_name=next((n for n in uploads if n.endswith('.csv')),None)
assert csv_name, 'Upload labelled_features.csv'
df=pd.read_csv(csv_name)
required=FEATURES+['window_id','video_id','pair_a','pair_b','end_s','source_group','label','evidence']
assert set(required).issubset(df.columns), 'Use labelled_features.csv, not annotations.csv'
print('Rows:',len(df),'videos:',df.video_id.nunique(),'source groups:',df.source_group.nunique())
display(df.head())
display(df[required].isna().sum().rename('missing').to_frame())
print('Duplicate windows:',df.window_id.duplicated().sum())
assert not df.window_id.duplicated().any(), 'Resolve duplicates'
assert df.label.isin(STATES).all(), 'Unexpected state name'
display(pd.crosstab(df.source_group,df.label).reindex(columns=STATES,fill_value=0))


In [ ]:
# 6. EDA: class support, source distribution and feature summaries.
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(14,4))
df.label.value_counts().reindex(STATES,fill_value=0).plot.bar(ax=axes[0],color=['#2563eb','#eab308','#f97316','#dc2626'])
axes[0].set_title('Labelled windows by state'); axes[0].set_ylabel('Correlated windows')
df.groupby('source_group').size().plot.bar(ax=axes[1],color='#2563eb')
axes[1].set_title('Windows by original recording')
fig.tight_layout(); fig.savefig(OUTPUT/'eda_counts.png',dpi=160); plt.show()
display(df[FEATURES].describe().T)


## Predefine the held-out recordings
Set the groups before examining model scores. A functional floor is two training recordings and one test recording, with **all four states in both partitions**. This is not enough evidence of generalization by itself. If missing states prevent training, collect/review additional independent examples; do not fabricate labels or repeatedly search for the highest-scoring test split.

The model uses fixed parameters, class weights and seed 42. No tuning is performed on the held-out test set. Missing/non-finite features and inconsistent annotations fail validation. A median imputer remains inside the saved pipeline, although the current extractor should output complete finite features.


In [ ]:
# 7. Edit using EXACT source groups displayed above.
TEST_GROUPS=['CHANGE_TO_HELD_OUT_RECORDING']
assert set(TEST_GROUPS).issubset(set(df.source_group.astype(str))), 'Edit TEST_GROUPS'
df.to_csv(OUTPUT/'labelled_features.csv',index=False)
report=train_model(df,OUTPUT/'artifacts',TEST_GROUPS)
print('Macro F1:',round(report['macro_f1'],3))
print('Balanced accuracy:',round(report['balanced_accuracy'],3))
print('Quadratic weighted kappa:',round(report['quadratic_weighted_kappa'],3))
print('Majority baseline macro F1:',round(report['majority_baseline_macro_f1'],3))
display(pd.DataFrame(report['classification_report']).T)


In [ ]:
# 8. Confusion matrix from held-out predictions only.
import numpy as np
cm=np.array(report['confusion_matrix'])
fig,ax=plt.subplots(figsize=(8,7)); image=ax.imshow(cm,cmap='Blues')
ax.set_xticks(range(4),STATES,rotation=40,ha='right'); ax.set_yticks(range(4),STATES)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Held-out source recordings')
for i in range(4):
    for j in range(4): ax.text(j,i,str(cm[i,j]),ha='center',va='center')
fig.colorbar(image,ax=ax); fig.tight_layout(); fig.savefig(OUTPUT/'confusion_matrix.png',dpi=180); plt.show()
importance=pd.read_csv(OUTPUT/'artifacts/feature_importance.csv')
importance.sort_values('importance').plot.barh(x='feature',y='importance',legend=False,figsize=(8,6))
plt.title('Feature importance (not causal evidence)'); plt.tight_layout(); plt.show()


In [ ]:
# 9. Download artifacts and copy them into the local package artifacts/ folder.
# Keep environment versions aligned. Only load your own trusted pickle files.
import shutil
shutil.copy('installed_versions.txt',OUTPUT/'artifacts/installed_versions.txt')
for name in ['risk_model.pkl','evaluation.json','split_manifest.csv','test_predictions.csv','feature_importance.csv','installed_versions.txt']:
    files.download(str(OUTPUT/'artifacts'/name))


## PP1 interpretation
Report actual recordings/windows, class support and measured scores. Overlapping windows are correlated; they are not unique events. Vote shares are uncalibrated, not collision probabilities. Image coordinates are not metres/km/h or calibrated TTC/PET.

A five-second window may include a collision, so classification does not establish advance prediction. Next stages: reviewed conflict-onset labels and early-warning lead time, camera-held-out evaluation, false alarms/hour, calibration, temporal model comparisons, ablation and full pipeline latency. Geographic aggregation needs verified camera coordinates. Three-wheelers, perspective, occlusion and identity errors remain pilot limitations.
